In [16]:
import skimage
from os import walk
import cv2
import os
from skimage import io
from skimage.io import imread_collection, imshow, show, imread
from skimage.transform import resize
import matplotlib.pyplot as plt
from skimage import data, color
import numpy as np
from numpy import array
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import image
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
import json
from collections import namedtuple
import csv

In [17]:
Rectangle = namedtuple('Rectangle', 'xmin ymin xmax ymax')
def area(a,b):  # returns None if rectangles don't intersect
    dx = min(a.xmax, b.xmax) - max(a.xmin, b.xmin)
    dy = min(a.ymax, b.ymax) - max(a.ymin, b.ymin)
    if (dx>=0) and (dy>=0):
        return dx*dy

#if no label is present for a slice, the slice is skipped
def slice_img(img, slice_size, json_filepath):
    img_slices= []
    label_list=[]
    h,w = img.shape[0], img.shape[1]
    #take grayscale
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #image padding
    padded_img = np.pad(gray_img, ((0, slice_size-h%slice_size), (0, slice_size-w%slice_size)),
                        'constant', constant_values=(255, 255))
    h,w = padded_img.shape[0], padded_img.shape[1]
    with open(json_filepath,"r") as f:
        json_dict = json.load(f)
    for i in range(0,h,slice_size):
        for j in range(0,w,slice_size):
            x1=i
            y1=j
            x2=i+slice_size
            y2=j+slice_size
            max_area=0
            label=""
            for key in json_dict:
                coord = json_dict[key]['coordinates']
                a1=coord[0][0]
                b1=coord[0][1]
                a2=coord[1][0]
                b2=coord[1][1]
                a=Rectangle(x1,y1,x2,y2)
                b=Rectangle(a1,b1,a2,b2)
                if(area(a,b)!=None and area(a,b) > max_area):
                    max_area=area(a,b)
                    label=json_dict[key]['label']
            if(label!=""):
                label_list.append(label)
                img_slices.append(padded_img[x1:x2, y1:y2])
    #print(len(label_list), len(img_slices))
    return img_slices, label_list


In [18]:
#preparing slices and labels
#store them in a csv file

#SlicedImgList is the list of
def slice_store(csv_filepath, slicedImgList, labelList):
    with open(csv_filepath, 'a', newline='') as newFile:
        #print(csv_filepath)
        newFileWriter = csv.writer(newFile)
        for array, label in zip(slicedImgList, labelList):
            flattened_array = array.reshape(1, -1)
            newFileWriter.writerow([flattened_array.tolist(), label])


def create_store_slices(csv_filepath, img_list, json_dir, slice_size):
    for img, json_filepath in zip(img_list, json_dir):
        slicedImgList, labelList = slice_img(img, slice_size, json_filepath)
        slice_store(csv_filepath, slicedImgList, labelList)
    

In [19]:
f= []
img_dir = []
json_dir = []
myImgPath = r"C:\Users\HP\Desktop\Model\dataset\Images"
for (dirpath, dirnames, filenames) in walk(myImgPath):
    f.extend(filenames)
    break
myJsonPath = r"C:\Users\HP\Desktop\Model\dataset\Json"
for each in f:
    img_dir.append(os.path.join(myImgPath,each))
    json_dir.append(os.path.join(myJsonPath, each.replace('.tif', '.json')))

In [20]:
img_list = []
for each_img in img_dir:
    img_list.append(cv2.imread(each_img))
#print(img_list)

In [21]:
slices = [20, 50, 80]
for slice_size in slices:
    csv_filepath = r'C:\Users\HP\Desktop\Model\csv\slices_'+str(slice_size)+'.csv'
    create_store_slices(csv_filepath, img_list, json_dir,slice_size)

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
data_csv_filepath = r'C:\Users\HP\Desktop\Model\csv\slices_'+str(slice_size)+'.csv'
print(data_csv_filepath)
#Train the model
#retrieve data from csv file
for slice_size in slices:
    X= [] 
    Y= []
    test_size = 0.2
    with open(data_csv_filepath, newline='') as dataFile:
            reader = list(csv.reader(dataFile))
            for row in reader:
                np_array = np.array(list(row[0][2:-2].split(',')), dtype=np.float)
                X.append(np_array)
                Y.append(row[1])
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = test_size, random_state = 36)

    clf=RandomForestClassifier(n_estimators=100)

    #Train the model using the training sets y_pred=clf.predict(X_test)
    clf.fit(X_train,Y_train)

    Y_pred=clf.predict(X_test)

    #printing accuracy
    print("Accuracy:",metrics.accuracy_score(Y_test, Y_pred))